# Country & Policy Explorer v2

Produces `docs/plots/country_policy_explorer_v2.html` — a fully interactive standalone chart with:
- Searchable tag/chip country picker (max 3 countries, 186 available)
- Dual-axis: pick any 2 parameters independently
- Solid lines = left Y-axis, dashed lines = right Y-axis

Data: `epidemiology.csv` + `oxford-government-response.csv` merged on `(date, location_key)`.

In [1]:
import pandas as pd
import numpy as np
import json
import math
import pathlib
import pycountry

print('Loading epidemiology.csv...')
epi = pd.read_csv('../data/epidemiology.csv', parse_dates=['date'])
print(f'  {len(epi):,} rows')

print('Loading oxford-government-response.csv...')
policy = pd.read_csv('../data/oxford-government-response.csv', parse_dates=['date'])
print(f'  {len(policy):,} rows')

Loading epidemiology.csv...
  12,525,825 rows
Loading oxford-government-response.csv...
  303,969 rows


In [2]:
# Keep only top-level country codes — drop sub-national regions (contain underscore)
epi    = epi[~epi['location_key'].astype(str).str.contains('_', na=True)]
policy = policy[~policy['location_key'].astype(str).str.contains('_', na=True)]

df = pd.merge(epi, policy, on=['date', 'location_key'], how='inner')
df = df.sort_values(['location_key', 'date']).reset_index(drop=True)

print(f'Merged: {len(df):,} rows, {df["location_key"].nunique()} countries')

Merged: 170,140 rows, 186 countries


In [3]:
NUM_COLS = [c for c in df.columns if c not in ('date', 'location_key')]

# Clip negative values (data artefacts) before rolling
for col in ['new_confirmed', 'new_deceased', 'new_recovered', 'new_tested']:
    if col in df.columns:
        df[col] = df[col].clip(lower=0)

print('Computing 7-day rolling averages...')
df[NUM_COLS] = (
    df.groupby('location_key')[NUM_COLS]
      .transform(lambda s: s.rolling(7, min_periods=1).mean())
)
df[NUM_COLS] = df[NUM_COLS].round(2)
df['date_str'] = df['date'].dt.strftime('%Y-%m-%d')
print('Done.')

Computing 7-day rolling averages...
Done.


In [4]:
def iso2_to_name(code):
    try:
        return pycountry.countries.get(alpha_2=code).name
    except Exception:
        return code

countries = sorted(df['location_key'].unique())
country_names = {c: iso2_to_name(c) for c in countries}

print(f'{len(country_names)} countries loaded')
for k in ['DK', 'DE', 'GB', 'US', 'FR', 'JP']:
    if k in country_names:
        print(f'  {k} -> {country_names[k]}')

186 countries loaded
  DK -> Denmark
  DE -> Germany
  GB -> United Kingdom
  US -> United States
  FR -> France
  JP -> Japan


In [5]:
def iso2_to_iso3(code):
    try:
        return pycountry.countries.get(alpha_2=code).alpha_3
    except Exception:
        return None

country_iso3 = {c: iso2_to_iso3(c) for c in countries if iso2_to_iso3(c)}

all_dates_list = sorted(df['date_str'].unique().tolist())
print(f'{len(country_iso3)} countries with ISO-3 mapping')
print(f'Date range: {all_dates_list[0]} → {all_dates_list[-1]} ({len(all_dates_list)} dates)')

185 countries with ISO-3 mapping
Date range: 2020-01-01 → 2022-07-26 (938 dates)


In [6]:
EPI_PARAMS = {
    'new_confirmed':        'New confirmed cases (7-day avg)',
    'new_deceased':         'New deaths (7-day avg)',
    'new_tested':           'New tests (7-day avg)',
    'cumulative_confirmed': 'Cumulative confirmed cases',
    'cumulative_deceased':  'Cumulative deaths',
    'cumulative_tested':    'Cumulative tests',
}

POLICY_PARAMS = {
    'stringency_index':              'Stringency index',
    'school_closing':                'School closing',
    'workplace_closing':             'Workplace closing',
    'stay_at_home_requirements':     'Stay-at-home requirements',
    'public_transport_closing':      'Public transport closing',
    'facial_coverings':              'Facial coverings',
    'testing_policy':                'Testing policy',
    'vaccination_policy':            'Vaccination policy',
    'restrictions_on_gatherings':    'Restrictions on gatherings',
    'cancel_public_events':          'Cancel public events',
    'income_support':                'Income support',
    'contact_tracing':               'Contact tracing',
    'international_travel_controls': 'International travel controls',
}

ALL_PARAMS    = {**EPI_PARAMS, **POLICY_PARAMS}
ALL_PARAMS    = {k: v for k, v in ALL_PARAMS.items()     if k in df.columns}
EPI_PARAMS    = {k: v for k, v in EPI_PARAMS.items()     if k in ALL_PARAMS}
POLICY_PARAMS = {k: v for k, v in POLICY_PARAMS.items()  if k in ALL_PARAMS}

print('Parameters:', list(ALL_PARAMS.keys()))

Parameters: ['new_confirmed', 'new_deceased', 'new_tested', 'cumulative_confirmed', 'cumulative_deceased', 'cumulative_tested', 'stringency_index', 'school_closing', 'workplace_closing', 'stay_at_home_requirements', 'public_transport_closing', 'facial_coverings', 'testing_policy', 'vaccination_policy', 'restrictions_on_gatherings', 'cancel_public_events', 'income_support', 'contact_tracing', 'international_travel_controls']


In [7]:
# Serialize as column-major JSON: { 'DK': { 'dates': [...], 'new_confirmed': [...], ... }, ... }
# Column-major is faster to access per-parameter in JavaScript and smaller than row-major.

def safe_list(series):
    return [
        None if (v is None or (isinstance(v, float) and (math.isnan(v) or math.isinf(v))))
        else v
        for v in series
    ]

print('Serializing...')
data_by_country = {}
for key, group in df.groupby('location_key'):
    entry = {'dates': group['date_str'].tolist()}
    for col in ALL_PARAMS:
        entry[col] = safe_list(group[col])
    data_by_country[key] = entry

sample_json = json.dumps(data_by_country, separators=(',', ':'))
print(f'JSON size: {len(sample_json) / 1e6:.1f} MB')

Serializing...
JSON size: 18.5 MB


In [8]:
def generate_html(data_by_country, country_names, epi_params, policy_params, country_iso3, all_dates):
    all_params         = {**epi_params, **policy_params}
    data_json          = json.dumps(data_by_country, separators=(',', ':'))
    country_names_json = json.dumps(country_names,   separators=(',', ':'))
    param_labels_json  = json.dumps(all_params,       separators=(',', ':'))
    country_iso3_json  = json.dumps(country_iso3,     separators=(',', ':'))
    all_dates_json     = json.dumps(all_dates,        separators=(',', ':'))

    def build_options(selected_key):
        lines = ['<optgroup label="── Epidemiological ──">']
        for k, v in epi_params.items():
            sel = ' selected' if k == selected_key else ''
            lines.append(f'  <option value="{k}"{sel}>{v}</option>')
        lines.append('</optgroup>')
        lines.append('<optgroup label="── Policy ──">')
        for k, v in policy_params.items():
            sel = ' selected' if k == selected_key else ''
            lines.append(f'  <option value="{k}"{sel}>{v}</option>')
        lines.append('</optgroup>')
        return '\n'.join(lines)

    opts_p1 = build_options('new_confirmed')
    opts_p2 = build_options('stringency_index')

    # JS template — placeholders are replaced below before writing to HTML
    js = """
const DATA          = DATA_JSON;
const COUNTRY_NAMES = COUNTRY_NAMES_JSON;
const PARAM_LABELS  = PARAM_LABELS_JSON;
const ALL_DATES     = ALL_DATES_JSON;
const COUNTRY_ISO3  = COUNTRY_ISO3_JSON;

const MAX_COUNTRIES = 3;
const COLORS = ['#6989f5', '#f07f8c', '#68d9a4'];
let selectedCountries = [];

const COUNT_PARAMS = new Set([
  'new_confirmed','new_deceased','new_tested',
  'cumulative_confirmed','cumulative_deceased','cumulative_tested'
]);

const POP_DATA = {
  AD:77265,AE:9770529,AF:38928341,AL:2877800,AO:32866268,AR:45195777,AT:9006398,
  AU:25499884,AW:106766,AZ:10139177,BA:3280815,BB:287371,BD:164689383,BE:11589616,
  BF:20903278,BG:6948445,BH:1701583,BI:11890781,BJ:12123198,BM:64184,BN:437483,
  BO:11673029,BR:212559409,BS:393248,BT:771612,BW:2351625,BY:9449323,BZ:397621,
  CA:37742157,CD:89561404,CF:4829764,CG:5518092,CH:8654618,CI:26378275,CL:19116209,
  CM:26545864,CN:1439323774,CO:50882884,CR:5094114,CU:11326616,CV:555988,CY:1207361,
  CZ:10708981,DE:83783945,DJ:988002,DK:5792203,DM:71991,DO:10847904,DZ:43851043,
  EC:17643060,EE:1326539,EG:102334403,ER:3546427,ES:46754783,ET:114963583,FI:5540718,
  FJ:896444,FO:48863,FR:65273512,GA:2225728,GB:67886004,GD:112519,GE:3989175,
  GH:31072945,GL:56367,GM:2416664,GN:13132792,GR:10423056,GT:17915567,GU:168783,
  GY:786559,HK:7496988,HN:9904608,HR:4105268,HT:11402533,HU:9660350,ID:273523621,
  IE:4937796,IL:8655541,IN:1380004385,IQ:40222503,IR:83992953,IS:341250,IT:60461828,
  JM:2961161,JO:10203140,JP:126476458,KE:53771296,KG:6524191,KH:16718971,KI:119446,
  KM:869595,KR:51269183,KW:4270563,KZ:18776707,LA:7275556,LB:6825442,LI:38137,
  LK:21413249,LR:5057677,LS:2142249,LT:2722291,LU:625976,LV:1886202,LY:6871287,
  MA:36910558,MC:39244,MD:4033963,MG:27691019,ML:20250834,MM:54409794,MN:3278292,
  MO:649342,MR:4649660,MT:441539,MU:1271767,MW:19129955,MX:128932753,MY:32365998,
  MZ:31255435,NE:24206636,NG:206139587,NI:6624554,NL:17134873,NO:5421242,NP:29136808,
  NZ:4822233,OM:4974992,PA:4314768,PE:32971846,PG:8947027,PH:109581085,PK:220892331,
  PL:37846605,PR:3194034,PS:4803269,PT:10196707,PY:7132530,QA:2881060,RO:19237682,
  RS:8737370,RU:145934460,RW:12952218,SA:34813867,SB:686878,SC:98340,SD:43849260,
  SE:10099270,SG:5850342,SI:2078938,SK:5459643,SL:7976985,SM:33938,SN:16743930,
  SO:15893219,SR:586634,SS:11193729,SV:6486201,SY:17500657,SZ:1160164,TD:16425859,
  TG:8278737,TH:69799978,TJ:9537642,TL:1318442,TM:6031187,TN:11818618,TO:105697,
  TR:84339067,TT:1399491,TW:23816775,TZ:59734213,UA:43733759,UG:45741000,
  US:331002647,UY:3473727,UZ:33469199,VE:28435940,VI:104578,VN:97338583,
  VU:307150,XK:1775378,YE:29825968,ZA:59308690,ZM:18383956,ZW:14862927
};

const PARAM_RANGE = {};
for (const param of Object.keys(PARAM_LABELS)) {
  const isCount = COUNT_PARAMS.has(param);
  let lo = Infinity, hi = -Infinity;
  for (const [iso2, cdata] of Object.entries(DATA)) {
    const arr = cdata[param];
    if (!arr) continue;
    const scale = isCount ? ((POP_DATA[iso2] || 1e6) / 100000) : 1;
    for (const v of arr) {
      if (v != null && !isNaN(v)) {
        const t = Math.log10(Math.max(v / scale, 0) + 1);
        if (t < lo) lo = t; if (t > hi) hi = t;
      }
    }
  }
  PARAM_RANGE[param] = { min: lo === Infinity ? 0 : lo, max: hi === -Infinity ? 1 : hi };
}

const ALL_COUNTRIES = Object.entries(COUNTRY_NAMES).sort((a, b) => a[1].localeCompare(b[1]));

const searchInput  = document.getElementById('country-search');
const dropdownEl   = document.getElementById('country-dropdown');
const chipsEl      = document.getElementById('chips');
const param1Select = document.getElementById('param1');
const param2Select = document.getElementById('param2');
const chartEl      = document.getElementById('chart');

// ── Theme handling ────────────────────────────────────────────────────
function isDark() {
  return document.documentElement.getAttribute('data-theme') === 'dark';
}

function getPlotlyTheme() {
  if (isDark()) {
    return {
      fontColor:   '#b0a8d0',
      titleColor:  '#f0eeff',
      gridColor:   'rgba(255,255,255,0.06)',
      lineColor:   'rgba(255,255,255,0.10)',
      hoverBg:     '#1a1a2e',
      hoverBorder: 'rgba(255,255,255,0.15)',
      hoverFont:   '#f0eeff',
      rsBg:        'rgba(255,255,255,0.06)',
      rsBorder:    'rgba(255,255,255,0.12)',
      rsActive:    'rgba(105,137,245,0.30)',
      rsFont:      '#f0eeff',
    };
  }
  return {
    fontColor:   '#6a6a8c',
    titleColor:  '#1a1a2e',
    gridColor:   'rgba(26,26,46,0.07)',
    lineColor:   'rgba(26,26,46,0.15)',
    hoverBg:     '#f5f0ff',
    hoverBorder: 'rgba(160,140,220,0.30)',
    hoverFont:   '#1a1a2e',
    rsBg:        'rgba(255,255,255,0.75)',
    rsBorder:    'rgba(180,170,220,0.40)',
    rsActive:    'rgba(192,216,240,0.60)',
    rsFont:      '#3a3a5c',
  };
}

function getMapTheme() {
  if (isDark()) {
    return { land: '#1a1832', ocean: '#0c0a18', border: '#2a2848',
             fontColor: '#b0a8d0', paperBg: 'rgba(0,0,0,0)' };
  }
  return { land: '#e8e4f5', ocean: '#ede8f5', border: '#c8c0e8',
           fontColor: '#6a6a8c', paperBg: 'rgba(0,0,0,0)' };
}

function applyTheme(theme) {
  document.documentElement.setAttribute('data-theme', theme);
  if (!isFirstDraw) redraw();
  if (!isFirstMapDraw) updateMap();
}

// Receive theme broadcasts from parent index.html
window.addEventListener('message', e => {
  if (e.data && e.data.covidTheme) applyTheme(e.data.covidTheme);
});

// Default: system preference
const systemTheme = window.matchMedia('(prefers-color-scheme: dark)').matches ? 'dark' : 'light';
document.documentElement.setAttribute('data-theme', systemTheme);

// ── Country picker ────────────────────────────────────────────────────
function renderDropdown(query) {
  const q = query.trim().toLowerCase();
  const matches = q.length === 0
    ? ALL_COUNTRIES.slice(0, 80)
    : ALL_COUNTRIES.filter(([k, n]) => n.toLowerCase().includes(q) || k.toLowerCase().startsWith(q)).slice(0, 60);
  dropdownEl.innerHTML = '';
  if (matches.length === 0) {
    dropdownEl.innerHTML = '<div class="dropdown-item disabled">No results</div>';
  } else {
    matches.forEach(([key, name]) => {
      const already = selectedCountries.includes(key);
      const full    = selectedCountries.length >= MAX_COUNTRIES && !already;
      const item = document.createElement('div');
      item.className = 'dropdown-item' + (already || full ? ' disabled' : '');
      item.textContent = name + (already ? ' ✓' : '');
      if (!already && !full) {
        item.addEventListener('mousedown', e => {
          e.preventDefault();
          addCountry(key);
          searchInput.value = '';
          dropdownEl.classList.remove('open');
        });
      }
      dropdownEl.appendChild(item);
    });
  }
  dropdownEl.classList.add('open');
}

searchInput.addEventListener('focus', () => renderDropdown(searchInput.value));
searchInput.addEventListener('input', () => renderDropdown(searchInput.value));
searchInput.addEventListener('blur',  () => { setTimeout(() => dropdownEl.classList.remove('open'), 150); });

function addCountry(key) {
  if (selectedCountries.includes(key)) return;
  if (selectedCountries.length >= MAX_COUNTRIES) return;
  selectedCountries.push(key);
  renderChips();
  redraw();
  if (mapReady) updateMap();
}

function removeCountry(key) {
  selectedCountries = selectedCountries.filter(k => k !== key);
  renderChips();
  redraw();
  if (mapReady) updateMap();
}

function renderChips() {
  chipsEl.innerHTML = '';
  selectedCountries.forEach((key, i) => {
    const chip = document.createElement('span');
    chip.className = `chip chip-${i}`;
    chip.innerHTML = `${COUNTRY_NAMES[key] || key} <button class="chip-remove" aria-label="Remove">&#x2715;</button>`;
    chip.querySelector('.chip-remove').addEventListener('click', () => removeCountry(key));
    chipsEl.appendChild(chip);
  });
}

param1Select.addEventListener('change', () => { redraw(); updateMap(); });
param2Select.addEventListener('change', redraw);

// ── Chart ─────────────────────────────────────────────────────────────
let isFirstDraw = true;

function redraw() {
  if (selectedCountries.length === 0) {
    if (!isFirstDraw) {
      Plotly.purge(chartEl);
      chartEl.innerHTML = '<div class="empty-state">Search for a country above to get started</div>';
      isFirstDraw = true;
    }
    return;
  }
  const p1 = param1Select.value;
  const p2 = param2Select.value;
  const th = getPlotlyTheme();
  const traces = [];
  selectedCountries.forEach((key, i) => {
    const country = DATA[key];
    if (!country) return;
    const color = COLORS[i];
    const name  = COUNTRY_NAMES[key] || key;
    traces.push({
      type: 'scatter', mode: 'lines',
      x: country.dates, y: country[p1],
      name: `${name} — ${PARAM_LABELS[p1]}`,
      yaxis: 'y',
      line: { color: color, width: 2 },
      hovertemplate: `<b>${name}</b> (${PARAM_LABELS[p1]})<br>%{x}: %{y:.2f}<extra></extra>`,
    });
    traces.push({
      type: 'scatter', mode: 'lines',
      x: country.dates, y: country[p2],
      name: `${name} — ${PARAM_LABELS[p2]} (right)`,
      yaxis: 'y2',
      line: { color: color, width: 2, dash: 'dot' },
      opacity: 0.75,
      hovertemplate: `<b>${name}</b> (${PARAM_LABELS[p2]})<br>%{x}: %{y:.2f}<extra></extra>`,
    });
  });
  const layout = {
    paper_bgcolor: 'rgba(0,0,0,0)',
    plot_bgcolor:  'rgba(0,0,0,0)',
    font:   { family: 'Inter, sans-serif', color: th.fontColor,  size: 12 },
    margin: { t: 20, r: 80, b: 100, l: 70 },
    hovermode: 'x unified',
    hoverlabel: {
      bgcolor: th.hoverBg, bordercolor: th.hoverBorder,
      font: { family: 'Inter, sans-serif', color: th.hoverFont, size: 12 }
    },
    xaxis: {
      gridcolor: th.gridColor, linecolor: th.lineColor, tickcolor: th.lineColor,
      rangeselector: {
        bgcolor: th.rsBg, bordercolor: th.rsBorder, activecolor: th.rsActive,
        font: { color: th.rsFont, size: 11 },
        buttons: [
          { count: 3,  label: '3M', step: 'month', stepmode: 'backward' },
          { count: 6,  label: '6M', step: 'month', stepmode: 'backward' },
          { count: 1,  label: '1Y', step: 'year',  stepmode: 'backward' },
          { step: 'all', label: 'All' }
        ]
      },
      rangeslider: { visible: false },
    },
    yaxis: {
      title: { text: PARAM_LABELS[p1], font: { color: th.titleColor, size: 12 } },
      gridcolor: th.gridColor, linecolor: th.lineColor, tickcolor: th.lineColor,
      zeroline: false,
    },
    yaxis2: {
      title: { text: PARAM_LABELS[p2], font: { color: th.titleColor, size: 12 } },
      overlaying: 'y', side: 'right',
      gridcolor: 'rgba(0,0,0,0)', linecolor: th.lineColor, tickcolor: th.lineColor,
      zeroline: false,
    },
    legend: {
      orientation: 'h', y: -0.28, x: 0,
      bgcolor: 'rgba(0,0,0,0)', font: { size: 11 }
    },
  };
  const config = {
    responsive: true, displaylogo: false,
    modeBarButtonsToRemove: ['lasso2d', 'select2d'],
  };
  if (isFirstDraw) { chartEl.innerHTML = ''; isFirstDraw = false; }
  Plotly.react(chartEl, traces, layout, config);
}

// ── World map ─────────────────────────────────────────────────────────
const mapEl        = document.getElementById('map');
const dateSliderEl = document.getElementById('date-slider');
const dateLabelEl  = document.getElementById('date-label');
const mapParamEl   = document.getElementById('map-param-label');

dateSliderEl.max   = ALL_DATES.length - 1;
dateSliderEl.value = ALL_DATES.length - 1;

let isFirstMapDraw = true;
let rafPending     = false;
let mapReady       = false;

function updateMap() {
  const idx     = parseInt(dateSliderEl.value, 10);
  const dateStr = ALL_DATES[idx];
  const p1      = param1Select.value;
  const th      = getMapTheme();

  dateLabelEl.textContent = dateStr;
  mapParamEl.textContent  = 'Showing: ' + PARAM_LABELS[p1];

  const isCount = COUNT_PARAMS.has(p1);
  const locs = [], zVals = [], texts = [];
  for (const [iso2, iso3] of Object.entries(COUNTRY_ISO3)) {
    const cdata = DATA[iso2];
    if (!cdata) continue;
    const di  = cdata.dates.indexOf(dateStr);
    const val = di >= 0 ? cdata[p1][di] : null;
    if (val === null || val === undefined) continue;
    const scale = isCount ? ((POP_DATA[iso2] || 1e6) / 100000) : 1;
    const vScaled = val / scale;
    const vLog = Math.log10(Math.max(vScaled, 0) + 1);
    locs.push(iso3);
    zVals.push(vLog);
    const label = isCount ? vScaled.toFixed(2) + ' per 100k' : val.toFixed(2);
    texts.push((COUNTRY_NAMES[iso2] || iso2) + ': ' + label);
  }

  const traces = [
    {
      type: 'choropleth', locationmode: 'ISO-3',
      locations: locs, z: zVals, text: texts,
      colorscale: 'YlOrRd', reversescale: true,
      zmin: PARAM_RANGE[p1].min,
      zmax: PARAM_RANGE[p1].max,
      showscale: true,
      marker: { line: { color: th.border, width: 0.5 } },
      colorbar: (() => {
        const logMax = PARAM_RANGE[p1].max;
        const cands = isCount
          ? [0,0.5,1,2,5,10,20,50,100,200,500,1000,2000,5000,10000,50000]
          : [0,0.5,1,2,5,10,20,50,100,200,500];
        const tOrig = cands.filter(v => Math.log10(v+1) <= logMax + 0.01);
        return {
          thickness: 12, len: 0.6, x: 1.01,
          tickvals: tOrig.map(v => Math.log10(v+1)),
          ticktext: tOrig.map(v => v >= 1000 ? Math.round(v/1000)+'k' : v < 1 ? v.toFixed(1) : String(Math.round(v))),
          tickfont: { family: 'Inter, sans-serif', color: th.fontColor, size: 10 },
          outlinewidth: 0,
        };
      })(),
      hovertemplate: '%{text}<extra></extra>',
    },
  ];

  // One trace per selected country so each gets its own border color
  selectedCountries.forEach((iso2, i) => {
    const iso3 = COUNTRY_ISO3[iso2];
    if (!iso3) return;
    traces.push({
      type: 'choropleth', locationmode: 'ISO-3',
      locations: [iso3],
      z: [0],
      showscale: false,
      colorscale: [[0, 'rgba(0,0,0,0)'], [1, 'rgba(0,0,0,0)']],
      marker: { line: { color: COLORS[i], width: 2.5 } },
      hoverinfo: 'skip',
      showlegend: false,
    });
  });

  const layout = {
    uirevision: 'map',
    paper_bgcolor: th.paperBg,
    margin: { t: 0, r: 0, b: 0, l: 0 },
    geo: {
      showframe: false, showcoastlines: false,
      showcountries: true, countrycolor: th.border,
      landcolor: th.land, bgcolor: th.ocean,
      projection: { type: 'natural earth' },
    },
  };

  const config = {
    responsive: true, displaylogo: false,
    modeBarButtonsToRemove: ['lasso2d', 'select2d'],
  };

  isFirstMapDraw = false;
  Plotly.react(mapEl, traces, layout, config);
}

dateSliderEl.addEventListener('input', () => {
  if (rafPending) return;
  rafPending = true;
  requestAnimationFrame(() => { rafPending = false; updateMap(); });
});

// ── Play / pause ───────────────────────────────────────────────────────
const playBtnEl = document.getElementById('play-btn');
let _playTimer  = null;
const PLAY_STEP_DAYS = 7;   // advance one week per tick
const PLAY_INTERVAL  = 150; // ms between ticks (~6.7 fps)

function _playStep() {
  let idx = parseInt(dateSliderEl.value, 10) + PLAY_STEP_DAYS;
  if (idx >= ALL_DATES.length) {
    idx = ALL_DATES.length - 1;
    dateSliderEl.value = idx;
    updateMap();
    clearInterval(_playTimer);
    _playTimer = null;
    playBtnEl.textContent = '▶';
    return;
  }
  dateSliderEl.value = idx;
  updateMap();
}

playBtnEl.addEventListener('click', () => {
  if (_playTimer) {
    clearInterval(_playTimer);
    _playTimer = null;
    playBtnEl.textContent = '▶';
  } else {
    if (parseInt(dateSliderEl.value, 10) >= ALL_DATES.length - 1) {
      dateSliderEl.value = 0;
      updateMap();
    }
    playBtnEl.textContent = '⏸';
    _playTimer = setInterval(_playStep, PLAY_INTERVAL);
  }
});

['DK', 'DE', 'GB'].forEach(k => { if (DATA[k]) addCountry(k); });
mapReady = true;
updateMap();
"""

    # Inject Python data into JS — order is safe since placeholders are unique
    js = js.replace('DATA_JSON',          data_json)
    js = js.replace('COUNTRY_NAMES_JSON', country_names_json)
    js = js.replace('PARAM_LABELS_JSON',  param_labels_json)
    js = js.replace('ALL_DATES_JSON',     all_dates_json)
    js = js.replace('COUNTRY_ISO3_JSON',  country_iso3_json)

    arrow_light = "url(\"data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' width='12' height='8' viewBox='0 0 12 8'%3E%3Cpath d='M1 1l5 5 5-5' stroke='%236a6a8c' stroke-width='1.5' fill='none' stroke-linecap='round'/%3E%3C/svg%3E\")"
    arrow_dark  = "url(\"data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' width='12' height='8' viewBox='0 0 12 8'%3E%3Cpath d='M1 1l5 5 5-5' stroke='%23b0a8d0' stroke-width='1.5' fill='none' stroke-linecap='round'/%3E%3C/svg%3E\")"

    return f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Country &amp; Policy Explorer</title>
  <link rel="preconnect" href="https://fonts.googleapis.com">
  <link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
  <link href="https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600&display=swap" rel="stylesheet">
  <script src="https://cdn.plot.ly/plotly-2.27.0.min.js" charset="utf-8"></script>
  <style>
    :root {{
      --body-bg:      #ede8f5;
      --glass-bg:     rgba(255,255,255,0.25);
      --glass-border: rgba(255,255,255,0.45);
      --text-dark:    #1a1a2e;
      --text-mid:     #3a3a5c;
      --text-light:   #6a6a8c;
      --dropdown-bg:  #f0eaf8;
      --input-bg:     rgba(255,255,255,0.55);
      --surface-hover: rgba(200,190,240,0.25);
      --chip-0-color: #3a52c0;
      --chip-1-color: #b83348;
      --chip-2-color: #1f7f55;
      --chip-0-bg:    rgba(105,137,245,0.12);
      --chip-1-bg:    rgba(240,127,140,0.12);
      --chip-2-bg:    rgba(104,217,164,0.12);
      --accent:       #6989f5;
      --radius: 16px;
      --blur: 14px;
    }}
    [data-theme="dark"] {{
      --body-bg:      #0c0a18;
      --glass-bg:     rgba(255,255,255,0.06);
      --glass-border: rgba(255,255,255,0.10);
      --text-dark:    #f0eeff;
      --text-mid:     #b0a8d0;
      --text-light:   #6a6288;
      --dropdown-bg:  #1a1a2e;
      --input-bg:     rgba(255,255,255,0.05);
      --surface-hover: rgba(255,255,255,0.10);
      --chip-0-color: #a0b4ff;
      --chip-1-color: #ffaab3;
      --chip-2-color: #8ef5c8;
      --chip-0-bg:    rgba(105,137,245,0.18);
      --chip-1-bg:    rgba(240,127,140,0.18);
      --chip-2-bg:    rgba(104,217,164,0.18);
    }}
    * {{ box-sizing: border-box; margin: 0; padding: 0; }}
    html, body {{
      font-family: 'Inter', sans-serif;
      background: var(--body-bg);
      color: var(--text-dark);
      min-height: 100vh;
      padding: 0;
      transition: background 0.4s, color 0.4s;
    }}
    .card {{
      background: var(--glass-bg);
      backdrop-filter: blur(var(--blur));
      -webkit-backdrop-filter: blur(var(--blur));
      border: 1px solid var(--glass-border);
      border-radius: var(--radius);
      padding: 20px 24px;
      margin-bottom: 16px;
      transition: background 0.4s, border-color 0.4s;
    }}
    h1 {{
      font-size: 1.15rem; font-weight: 600;
      margin-bottom: 18px; color: var(--text-dark);
      letter-spacing: -0.02em;
    }}
    .controls {{
      display: grid;
      grid-template-columns: 1fr 1fr;
      gap: 20px; align-items: start;
    }}
    @media (max-width: 640px) {{ .controls {{ grid-template-columns: 1fr; }} }}
    label {{
      display: block; font-size: 0.72rem; font-weight: 500;
      letter-spacing: 0.06em; text-transform: uppercase;
      color: var(--text-light); margin-bottom: 8px;
    }}
    .picker-section {{ grid-column: 1 / -1; }}
    .picker-box {{ position: relative; }}
    .picker-input {{
      width: 100%; background: var(--input-bg);
      border: 1px solid var(--glass-border); border-radius: 10px;
      color: var(--text-dark); font-family: 'Inter', sans-serif;
      font-size: 0.9rem; padding: 10px 14px; outline: none;
      transition: border-color 0.15s, background 0.4s;
    }}
    .picker-input::placeholder {{ color: var(--text-light); }}
    .picker-input:focus {{ border-color: var(--accent); }}
    .dropdown-list {{
      position: absolute; top: calc(100% + 6px); left: 0; right: 0;
      background: var(--dropdown-bg); border: 1px solid var(--glass-border);
      border-radius: 10px; max-height: 220px; overflow-y: auto;
      z-index: 100; display: none;
      box-shadow: 0 8px 32px rgba(0,0,0,0.18);
    }}
    .dropdown-list.open {{ display: block; }}
    .dropdown-item {{
      padding: 9px 14px; cursor: pointer;
      font-size: 0.875rem; color: var(--text-dark);
      transition: background 0.1s;
    }}
    .dropdown-item:hover {{ background: var(--surface-hover); }}
    .dropdown-item.disabled {{ opacity: 0.35; cursor: default; }}
    .chips {{ display: flex; flex-wrap: wrap; gap: 8px; margin-top: 10px; min-height: 34px; }}
    .chip {{
      display: inline-flex; align-items: center; gap: 6px;
      padding: 5px 12px; border-radius: 20px;
      font-size: 0.82rem; font-weight: 500; border: 1px solid;
      transition: background 0.3s, color 0.3s;
    }}
    .chip-0 {{ background: var(--chip-0-bg); border-color: rgba(105,137,245,0.45); color: var(--chip-0-color); }}
    .chip-1 {{ background: var(--chip-1-bg); border-color: rgba(240,127,140,0.45); color: var(--chip-1-color); }}
    .chip-2 {{ background: var(--chip-2-bg); border-color: rgba(104,217,164,0.45); color: var(--chip-2-color); }}
    .chip-remove {{
      cursor: pointer; opacity: 0.7; font-size: 1rem; line-height: 1;
      background: none; border: none; color: inherit; padding: 0;
    }}
    .chip-remove:hover {{ opacity: 1; }}
    .picker-hint {{ font-size: 0.75rem; color: var(--text-light); margin-top: 6px; }}
    select {{
      width: 100%; background: var(--input-bg);
      border: 1px solid var(--glass-border); border-radius: 10px;
      color: var(--text-dark); font-family: 'Inter', sans-serif;
      font-size: 0.875rem; padding: 10px 14px; outline: none;
      cursor: pointer; appearance: none;
      background-image: {arrow_light};
      background-repeat: no-repeat; background-position: right 14px center;
      padding-right: 36px; transition: border-color 0.15s, background 0.4s;
    }}
    [data-theme="dark"] select {{
      background-image: {arrow_dark};
    }}
    select:focus {{ border-color: var(--accent); }}
    optgroup {{ color: var(--text-light); font-style: normal; }}
    option {{ background: var(--dropdown-bg); color: var(--text-dark); }}
    .axis-hint {{ font-size: 0.72rem; color: var(--text-light); margin-top: 6px; }}
    .dot-line {{
      display: inline-block; width: 24px; height: 2px;
      border-top: 2px dashed var(--text-light); vertical-align: middle; margin-right: 4px;
    }}
    .solid-line {{
      display: inline-block; width: 24px; height: 2px;
      border-top: 2px solid var(--text-light); vertical-align: middle; margin-right: 4px;
    }}
    #chart {{ width: 100%; height: 520px; }}
    .empty-state {{
      display: flex; align-items: center; justify-content: center;
      height: 520px; color: var(--text-light); font-size: 0.95rem;
    }}
    .map-controls {{ margin-bottom: 14px; }}
    .slider-row {{
      display: flex; align-items: center; gap: 12px; margin-top: 8px;
    }}
    #date-slider {{
      flex: 1; accent-color: var(--accent); cursor: pointer;
    }}
    #date-label {{
      font-size: 0.82rem; font-weight: 500; color: var(--text-mid);
      white-space: nowrap; min-width: 90px;
    }}
    #map-param-label {{
      font-size: 0.75rem; color: var(--text-light); margin-top: 6px;
    }}
    #map {{ width: 100%; height: 440px; }}
  </style>
</head>
<body>
<div class="card">
  <h1>Country &amp; Policy Explorer</h1>
  <div class="controls">
    <div class="picker-section">
      <label>Countries (up to 3)</label>
      <div class="picker-box">
        <input id="country-search" class="picker-input" type="text"
               placeholder="Search countries..." autocomplete="off">
        <div id="country-dropdown" class="dropdown-list"></div>
      </div>
      <div id="chips" class="chips"></div>
      <p class="picker-hint">Select up to 3 countries to compare</p>
    </div>
    <div>
      <label for="param1">Y-axis 1 — left axis</label>
      <select id="param1">{opts_p1}</select>
      <p class="axis-hint"><span class="solid-line"></span>Solid lines</p>
    </div>
    <div>
      <label for="param2">Y-axis 2 — right axis</label>
      <select id="param2">{opts_p2}</select>
      <p class="axis-hint"><span class="dot-line"></span>Dashed lines</p>
    </div>
  </div>
</div>
<div class="card" style="padding: 8px;">
  <div id="chart"><div class="empty-state">Search for a country above to get started</div></div>
</div>
<div class="card">
  <div class="map-controls">
    <label for="date-slider">World map — snapshot date</label>
    <div class="slider-row">
      <input type="range" id="date-slider" min="0" step="1">
      <span id="date-label"></span>
    </div>
    <p id="map-param-label"></p>
  </div>
  <div id="map"></div>
</div>
<script>{js}</script>
</body>
</html>"""


print('generate_html defined.')


generate_html defined.


In [9]:
html = generate_html(
    data_by_country, country_names,
    EPI_PARAMS, POLICY_PARAMS,
    country_iso3, all_dates_list,
)

out = pathlib.Path('../docs/plots/country_policy_explorer_v2.html')
out.write_text(html, encoding='utf-8')

print(f'Written: {out}')
print(f'Size:    {out.stat().st_size / 1e6:.1f} MB')

Written: ../docs/plots/country_policy_explorer_v2.html
Size:    18.5 MB
